# NAMAA V7 — Actual 60-second Egyptian speech proof
Reuses V6 verified environment and checkpoint. Per-segment validation/caching, stitches 0.18-sec silent gaps, creates WAV, MP3, and JSON report. Stop on >=60 seconds audio, not a fixed number of clips. FREE Kaggle T4, no Project 79 VPS.

In [ ]:
import os,subprocess
from pathlib import Path
venv="/kaggle/working/namaa_venv_v6/bin/python"
assert Path(venv).exists(),"V6 previously successful venv is missing: need to restore V6 environment first"
print("REUSING_VERIFIED_ENV",venv,flush=True)
env=os.environ.copy()
env.pop("PYTHONPATH",None)
env.pop("PYTHONHOME",None)
env["PYTHONNOUSERSITE"]="1"
env["HF_HOME"]="/kaggle/working/namaa_hf_cache"
src=Path("/kaggle/working/namaa_produce_v7.py")
src.write_text("import os,time,json,subprocess\nfrom pathlib import Path\nimport torch,torchaudio\nfrom huggingface_hub import snapshot_download\nfrom safetensors.torch import load_file\nfrom chatterbox.mtl_tts import ChatterboxMultilingualTTS\n\nroot=Path(\"/kaggle/working\")\nsegments_dir=root/\"namaa_mansa_segments\"\nsegments_dir.mkdir(parents=True,exist_ok=True)\ntarget_seconds=60.0\nsentences=[\n\"تخيّل إنك تمشي في شارع، وتلاقي راجل بيوزّع دهب، وكأنه بيوزّع حلوى على الأطفال.\",\n\"الراجل ده مش شخصية من فيلم خيالي. ده مانسا موسى، حاكم إمبراطورية مالي زمان.\",\n\"في القرن الرابع عشر، إمبراطورية مالي كانت من أهم مراكز التجارة، وخصوصا تجارة الدهب والملح.\",\n\"مانسا موسى اشتهر بثروته الكبيرة، لكن اللي خلّى اسمه يوصل للعالم كله كانت رحلة الحج.\",\n\"سنة ألف تلتمية أربعة وعشرين، خرج في رحلة طويلة من غرب أفريقيا، لحد مكة المكرمة.\",\n\"القافلة بتاعته كانت ضخمة، وفيها ناس كتير وهدايا وكمية كبيرة من الدهب.\",\n\"ولما مرّ بالقاهرة، حكايات المؤرخين بتقول إنه وزّع دهب على ناس كتير.\",\n\"الدهب لما يزيد فجأة في السوق، قيمته ممكن تقل، ودي فكرة اقتصادية مهمة جدا.\",\n\"وعشان كده، قصته بتتقال كتير لما الناس تتكلم عن الأسعار والتضخّم.\",\n\"لكن خد بالك، مش كل رقم منتشر عن ثروته حقيقي أو ممكن نثبته بدقة.\",\n\"زمان ماكانش فيه حسابات بنكية وتقارير مالية بالشكل اللي نعرفه دلوقتي.\",\n\"فمن الصعب نحول ثروة ملك عاش من سبعمية سنة لرقم بالدولار النهارده.\",\n\"الأكيد إن مانسا موسى ساب أثر في التاريخ، مش بسبب الدهب بس، لكن بسبب النفوذ والتجارة.\",\n\"وفي تمبكتو، ازدهرت مراكز العلم والمعرفة، وبقى اسم مالي معروف في مناطق بعيدة.\",\n\"السؤال بقى، لو معاك دهب كتير جدا، هل ده لوحده يخليك أغنى واحد في العالم؟\",\n\"يمكن الإجابة الحقيقية مش في حجم الدهب، لكن في القوة اللي الثروة كانت بتديها لصاحبها.\",\n\"دي كانت حكاية مانسا موسى، ولو عايزين قصص اقتصاد وتاريخ أكتر، استنونا في الحكاية الجاية.\"\n]\nprint(\"V7_GPU\",torch.cuda.is_available(),torch.cuda.get_device_name(0),flush=True)\nassert torch.cuda.is_available()\nstart=time.monotonic()\nexisting=sorted(segments_dir.glob(\"segment_*.wav\"))\nprint(\"PREEXISTING_SEGMENTS\",len(existing),flush=True)\nengine=ChatterboxMultilingualTTS.from_pretrained(device=\"cuda\")\nweights=snapshot_download(\"NAMAA-Space/NAMAA-Egyptian-TTS\",allow_patterns=[\"t3_mtl23ls_v2.safetensors\"])\nengine.t3.load_state_dict(load_file(str(Path(weights)/\"t3_mtl23ls_v2.safetensors\"),device=\"cpu\"),strict=True)\nengine.t3.to(\"cuda\").eval()\nprint(\"MODEL_READY\",round(time.monotonic()-start,2),flush=True)\naudio_parts=[]\ndata=[]\ngenerated_duration=0.0\nfor ix, sentence in enumerate(sentences,1):\n    path=segments_dir/(\"segment_%02d.wav\"%ix)\n    before=time.monotonic()\n    if path.exists() and path.stat().st_size>2000:\n        wav,sr=torchaudio.load(str(path))\n        assert sr==engine.sr\n        status=\"CACHED\"\n    else:\n        wav=engine.generate(sentence,language_id=\"ar\")\n        torchaudio.save(str(path),wav.detach().cpu(),engine.sr)\n        status=\"GENERATED\"\n    seconds=round(wav.shape[-1]/engine.sr,2)\n    audio_parts.append(wav.detach().cpu())\n    data.append({\"idx\":ix,\"text\":sentence,\"audio_seconds\":seconds,\"status\":status,\"file\":str(path),\"run_seconds\":round(time.monotonic()-before,2)})\n    generated_duration+=seconds\n    print(\"SEGMENT\",ix,status,\"AUDIO_SEC\",seconds,\"TOTAL_AUDIO_SEC\",round(generated_duration,1),\"RUN_SEC\",data[-1][\"run_seconds\"],flush=True)\n    if generated_duration + max(0,len(audio_parts)-1)*0.18 >= target_seconds:\n        break\npause=torch.zeros((1,round(engine.sr*0.18)))\ncombined=torch.cat([part for ix,w in enumerate(audio_parts) for part in ([w] if ix==0 else [pause,w])],dim=-1)\nout=root/\"namaa_mansa_60s.wav\"\ntorchaudio.save(str(out),combined,engine.sr)\nmp3=root/\"namaa_mansa_60s.mp3\"\nproc=subprocess.run([\"ffmpeg\",\"-y\",\"-hide_banner\",\"-loglevel\",\"error\",\"-i\",str(out),\"-codec:a\",\"libmp3lame\",\"-qscale:a\",\"3\",str(mp3)],capture_output=True,text=True)\nassert proc.returncode==0,proc.stderr\nreport={\"source\":\"NAMAA-Space/NAMAA-Egyptian-TTS\",\"gpu\":torch.cuda.get_device_name(0),\"sample_rate\":engine.sr,\"audio_seconds\":round(combined.shape[-1]/engine.sr,2),\"total_seconds\":round(time.monotonic()-start,2),\"segments\":data,\"wav_size\":out.stat().st_size,\"mp3_size\":mp3.stat().st_size,\"wav\":str(out),\"mp3\":str(mp3)}\n(root/\"namaa_mansa_report.json\").write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding=\"utf-8\")\nprint(\"PRODUCTION_SUCCESS\",json.dumps({k:v for k,v in report.items() if k!=\"segments\"},ensure_ascii=False),flush=True)\n",encoding="utf-8")
print("RUN_V7_IN_SAME_FRESH_PYTHON_PROCESS",flush=True)
subprocess.run([venv,"-u",str(src)],env=env,check=True,timeout=900)
